# Global Fundamental Feature Families EDA

This notebook builds global fundamental feature families from:

- FMP daily-adjusted valuation features
- FMP statement-level market-cap ratio families
- FinanceToolkit-style FMP-derived ratio and growth families

pandas-ta-classic technical families are intentionally excluded from this notebook. Every family is capped at 50 columns before predictive-power evaluation using coverage and cross-sectional variance only, not target performance.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

_current = Path.cwd().resolve()
_repo_candidates = [_current, *_current.parents]
REPO_ROOT = next((path for path in _repo_candidates if (path / "quant_warehouse").exists() and (path / "pyproject.toml").exists()), _current)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_warehouse import Warehouse
from quant_warehouse.research_tools import (
    FamilyEvaluationConfig,
    build_fundamental_feature_panel,
    cap_features_by_quality,
    evaluate_feature_families,
    screen_fmp_equity_universe,
)

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 300)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=1_000_000_000_000,
    country="US",
    exchanges=("NASDAQ", "NYSE", "AMEX"),
    start_date="2018-01-01",
    horizons=(20, 60, 120),
    max_features_per_family=50,
)
ANALYSIS_LABEL = "OpenBB/FMP screened US >= $1T universe"
wh = Warehouse()


## Universe

Use the FMP screener through OpenBB, then retain only symbols with stored Quant Warehouse history for prices, historical market cap, statements, ratios, metrics, and growth sections.


In [2]:
symbols, raw_universe, eligibility, screener_source = screen_fmp_equity_universe(config, warehouse=wh)

display(Markdown(f"> Selected {len(symbols):,} symbols from `{ANALYSIS_LABEL}` via `{screener_source}`."))
display(raw_universe.sort_values("market_cap", ascending=False) if "market_cap" in raw_universe.columns else raw_universe)
display(eligibility.sort_values("screen_market_cap", ascending=False) if "screen_market_cap" in eligibility.columns else eligibility)
display(eligibility["reason"].value_counts().rename_axis("reason").reset_index(name="symbols"))


> Selected 14 symbols from `OpenBB/FMP screened US >= $1T universe` via `catalog:fmp:fallback_after_OpenBBError+openbb:fmp`.

,symbol,name,market_cap,exchange,country,sector,industry,beta,price,last_annual_dividend,volume,exchange_name,is_etf,is_fund,actively_trading
0,NVDA,NVIDIA Corporation,"4,819,979,000,000.0000",NASDAQ,US,Technology,Semiconductors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL,Apple Inc.,"4,322,488,870,800.0000",NASDAQ,US,Technology,Consumer Electronics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GOOGL,Alphabet Inc.,"4,186,397,454,516.0898",NASDAQ,US,Communication Services,Internet Content & Information,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GOOG,Alphabet Inc.,"4,185,792,711,001.4399",NASDAQ,US,Communication Services,Internet Content & Information,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MSFT,Microsoft Corporation,"2,777,787,114,200.0000",NASDAQ,US,Technology,Software - Infrastructure,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AMZN,"Amazon.com, Inc.","2,518,344,681,000.0000",NASDAQ,US,Consumer Cyclical,Specialty Retail,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SPCX,Space Exploration Technologies Corp,"2,020,744,245,407.0000",NASDAQ,US,Technology,Telecommunications Services,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,AVGO,Broadcom Inc.,"1,808,594,037,000.0000",NASDAQ,US,Technology,Semiconductors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,TSLA,"Tesla, Inc.","1,433,220,309,200.0000",NASDAQ,US,Consumer Cyclical,Auto - Manufacturers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,META,"Meta Platforms, Inc.","1,427,101,580,946.6001",NASDAQ,US,Communication Services,Internet Content & Information,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,"4,819,979,000,000.0000"
1,AAPL,True,ok,"4,322,488,870,800.0000"
2,GOOGL,True,ok,"4,186,397,454,516.0898"
3,GOOG,True,ok,"4,185,792,711,001.4399"
4,MSFT,True,ok,"2,777,787,114,200.0000"
5,AMZN,True,ok,"2,518,344,681,000.0000"
6,SPCX,True,ok,"2,020,744,245,407.0000"
7,AVGO,True,ok,"1,808,594,037,000.0000"
8,TSLA,True,ok,"1,433,220,309,200.0000"
9,META,True,ok,"1,427,101,580,946.6001"


,reason,symbols
0,ok,14


## Build Raw Feature Panel

The shared helper builds FMP daily-adjusted valuation families, FMP statement-level `*_mcap` families, and FinanceToolkit-style ratio/growth families.


In [3]:
raw_panel, raw_feature_metadata, diagnostics_df, timings = build_fundamental_feature_panel(symbols, config, warehouse=wh)

display(diagnostics_df)
raw_family_counts = raw_feature_metadata.groupby(["source", "family"]).size().rename("raw_feature_count").reset_index().sort_values(["source", "raw_feature_count"], ascending=[True, False])
display(raw_family_counts)
print({
    "symbols": int(raw_panel["symbol"].nunique()),
    "rows": int(len(raw_panel)),
    "raw_features": int(raw_feature_metadata["feature"].nunique()),
    "raw_families": int(raw_feature_metadata["family"].nunique()),
    "raw_panel_build_seconds": round(timings["raw_panel_build_seconds"], 2),
})


,symbol,status,rows,features,seconds
0,AAPL,ok,2130,376,0.0791
1,AMZN,ok,2130,376,0.0682
2,AVGO,ok,2130,376,0.0631
3,BRK-A,ok,2130,376,0.2297
4,BRK-B,ok,2130,376,0.0743
5,GOOG,ok,2130,376,0.0764
6,GOOGL,ok,2130,376,0.0745
7,LLY,ok,2130,376,0.0713
8,META,ok,2130,376,0.0735
9,MSFT,ok,2130,376,0.0773


,source,family,raw_feature_count
0,financetoolkit,ft_growth_balance,56
1,financetoolkit,ft_growth_cash,50
2,financetoolkit,ft_growth_income,38
7,financetoolkit,ft_ratios_valuation,24
6,financetoolkit,ft_ratios_solvency,15
5,financetoolkit,ft_ratios_profitability,14
4,financetoolkit,ft_ratios_liquidity,7
3,financetoolkit,ft_ratios_efficiency,5
8,fmp,fmp_balance_mcap,55
9,fmp,fmp_cash_mcap,39


{'symbols': 14, 'rows': 27701, 'raw_features': 376, 'raw_families': 15, 'raw_panel_build_seconds': 1.23}


## Cap Each Family To 50 Columns

The cap uses non-target information only: valid observation coverage, average daily cross-sectional variance, then deterministic feature name order.


In [4]:
selected_features, feature_metadata, cap_quality = cap_features_by_quality(
    raw_panel,
    raw_feature_metadata,
    max_features=config.max_features_per_family,
)
feature_cols = feature_metadata["feature"].tolist()
panel = raw_panel.loc[:, ["date", "symbol", "close", *feature_cols, *[f"forward_return_{h}d" for h in config.horizons]]].copy()

cap_summary = (
    raw_feature_metadata.groupby(["source", "family"]).size().rename("raw_feature_count").reset_index()
    .merge(feature_metadata.groupby(["source", "family"]).size().rename("capped_feature_count").reset_index(), on=["source", "family"], how="left")
    .fillna({"capped_feature_count": 0})
)
cap_summary["capped_feature_count"] = cap_summary["capped_feature_count"].astype(int)
cap_summary["was_capped"] = cap_summary["raw_feature_count"].gt(cap_summary["capped_feature_count"])
display(cap_summary.sort_values(["source", "capped_feature_count"], ascending=[True, False]))
display(cap_quality.loc[cap_quality["selected"]].groupby("family").head(8))
print({"capped_features": len(feature_cols), "families": feature_metadata["family"].nunique(), "max_family_width": int(feature_metadata.groupby("family").size().max())})


,source,family,raw_feature_count,capped_feature_count,was_capped
0,financetoolkit,ft_growth_balance,56,50,True
1,financetoolkit,ft_growth_cash,50,50,False
2,financetoolkit,ft_growth_income,38,38,False
7,financetoolkit,ft_ratios_valuation,24,24,False
6,financetoolkit,ft_ratios_solvency,15,15,False
5,financetoolkit,ft_ratios_profitability,14,14,False
4,financetoolkit,ft_ratios_liquidity,7,7,False
3,financetoolkit,ft_ratios_efficiency,5,5,False
8,fmp,fmp_balance_mcap,55,50,True
9,fmp,fmp_cash_mcap,39,39,False


,family,feature,coverage,avg_xs_var,selected
0,fmp_balance_mcap,fmp_balance_mcap__total_liabilities_and_total_...,0.9996,0.2015,True
1,fmp_balance_mcap,fmp_balance_mcap__total_assets,0.9996,0.2015,True
2,fmp_balance_mcap,fmp_balance_mcap__total_non_current_assets,0.9996,0.1168,True
3,fmp_balance_mcap,fmp_balance_mcap__retained_earnings,0.9996,0.0700,True
4,fmp_balance_mcap,fmp_balance_mcap__total_equity,0.9996,0.0651,True
...,...,...,...,...,...
355,ft_ratios_valuation,ft_ratios_valuation__ev_to_free_cash_flow,1.0000,"11,708,899,329.0666",True
356,ft_ratios_valuation,ft_ratios_valuation__price_to_free_cash_flow_r...,1.0000,"11,707,597,410.6289",True
357,ft_ratios_valuation,ft_ratios_valuation__ev_to_operating_cash_flow,1.0000,"2,353,141,003.6385",True
358,ft_ratios_valuation,ft_ratios_valuation__book_value_per_share,1.0000,"1,120,480,047.7749",True


{'capped_features': 365, 'families': 15, 'max_family_width': 50}


## Family Evaluation

This run keeps the evaluator lightweight by computing family rank-IC summaries without top/bottom spread diagnostics. The point of this notebook is family structure and feature inventory first; deeper target diagnostics can be run later on selected families.


In [5]:
results_df, summary_by_family, best_by_horizon, stable_families, evaluation_seconds = evaluate_feature_families(
    panel,
    feature_metadata,
    horizons=config.horizons,
    min_observations=config.min_observations,
    include_spreads=False,
)

display(results_df.sort_values(["horizon", "mean_daily_rank_ic"], ascending=[True, False]).groupby("horizon").head(20))
display(summary_by_family)
display(best_by_horizon)
display(stable_families)
print({"evaluation_seconds": round(evaluation_seconds, 2), "feature_horizon_results": len(results_df)})


/home/jlee153232/PycharmProjects/quant-warehouse/quant_warehouse/research_tools/feature_family_eval.py:706: RuntimeWarning: invalid value encountered in divide
  daily_ic = numerator / denominator
/home/jlee153232/PycharmProjects/quant-warehouse/quant_warehouse/research_tools/feature_family_eval.py:706: RuntimeWarning: invalid value encountered in divide
  daily_ic = numerator / denominator
/home/jlee153232/PycharmProjects/quant-warehouse/quant_warehouse/research_tools/feature_family_eval.py:706: RuntimeWarning: invalid value encountered in divide
  daily_ic = numerator / denominator


,feature,horizon,mean_daily_rank_ic,median_daily_rank_ic,rank_ic_hit_rate,spread_bps,observations,family,source,source_column,expected_direction
262,ft_growth_income__growth_basic_earings_per_share,20,0.1355,0.1538,0.0761,NaN,3337,ft_growth_income,financetoolkit,income_growth.growth_basic_earings_per_share,higher_is_better
267,ft_growth_income__growth_diluted_earnings_per_...,20,0.1335,0.1538,0.0761,NaN,3337,ft_growth_income,financetoolkit,income_growth.growth_diluted_earnings_per_share,higher_is_better
263,ft_growth_income__growth_consolidated_net_income,20,0.1333,0.1538,0.0761,NaN,3337,ft_growth_income,financetoolkit,income_growth.growth_consolidated_net_income,higher_is_better
290,ft_growth_income__growth_research_and_developm...,20,0.1091,0.1264,0.0812,NaN,3337,ft_growth_income,financetoolkit,income_growth.growth_research_and_development_...,higher_is_better
160,fmp_income_mcap__weighted_average_shs_out,20,0.0991,0.1319,0.6103,NaN,27430,fmp_income_mcap,fmp,income.weighted_average_shs_out,higher_is_better
245,ft_growth_cash__growth_net_equity_issuance,20,0.0984,0.0989,0.0793,NaN,3337,ft_growth_cash,financetoolkit,cash_growth.growth_net_equity_issuance,higher_is_better
275,ft_growth_income__growth_gross_profit_margin,20,0.0977,0.1044,0.0761,NaN,3337,ft_growth_income,financetoolkit,income_growth.growth_gross_profit_margin,higher_is_better
161,fmp_income_mcap__weighted_average_shs_out_dil,20,0.0965,0.1319,0.6056,NaN,27430,fmp_income_mcap,fmp,income.weighted_average_shs_out_dil,higher_is_better
341,ft_ratios_valuation__book_value_per_share,20,0.0964,0.1264,0.6277,NaN,27430,ft_ratios_valuation,financetoolkit,ratios.book_value_per_share,lower_is_better
353,ft_ratios_valuation__graham_number,20,0.0939,0.1242,0.6286,NaN,25453,ft_ratios_valuation,financetoolkit,metrics.graham_number,lower_is_better


,horizon,source,family,features,mean_rank_ic,median_rank_ic,median_spread_bps,positive_ic_share
7,20,financetoolkit,ft_ratios_valuation,24,0.0213,-0.0074,NaN,0.4583
3,20,financetoolkit,ft_ratios_efficiency,5,0.0175,0.0231,NaN,0.8000
2,20,financetoolkit,ft_growth_income,38,0.0130,0.0023,NaN,0.5000
4,20,financetoolkit,ft_ratios_liquidity,7,0.0117,0.0087,NaN,0.5714
1,20,financetoolkit,ft_growth_cash,50,0.0061,0.0049,NaN,0.5600
5,20,financetoolkit,ft_ratios_profitability,14,0.0061,0.0081,NaN,0.5714
14,20,fmp,fmp_income_mcap,31,0.0001,-0.0009,NaN,0.4839
0,20,financetoolkit,ft_growth_balance,50,-0.0031,-0.0017,NaN,0.4200
9,20,fmp,fmp_cash_mcap,39,-0.0036,-0.0104,NaN,0.4103
8,20,fmp,fmp_balance_mcap,50,-0.0153,-0.0254,NaN,0.3600


,horizon,source,family,features,mean_rank_ic,median_rank_ic,median_spread_bps,positive_ic_share
0,20,financetoolkit,ft_ratios_valuation,24,0.0213,-0.0074,NaN,0.4583
1,60,financetoolkit,ft_ratios_valuation,24,0.0436,0.0129,NaN,0.5417
2,120,financetoolkit,ft_ratios_valuation,24,0.0611,0.0107,NaN,0.5833


,source,family,horizons,avg_rank_ic,min_rank_ic,avg_spread_bps,positive_horizons,avg_positive_ic_share,features
7,financetoolkit,ft_ratios_valuation,3,0.0420,0.0213,NaN,3,0.5278,24
3,financetoolkit,ft_ratios_efficiency,3,0.0264,0.0175,NaN,3,0.8000,5
2,financetoolkit,ft_growth_income,3,0.0229,0.0130,NaN,3,0.5702,38
4,financetoolkit,ft_ratios_liquidity,3,0.0166,0.0117,NaN,3,0.5238,7
5,financetoolkit,ft_ratios_profitability,3,0.0165,0.0061,NaN,3,0.7381,14
14,fmp,fmp_income_mcap,3,0.0046,0.0001,NaN,3,0.4624,31
1,financetoolkit,ft_growth_cash,3,0.0072,-0.0005,NaN,2,0.5333,50
9,fmp,fmp_cash_mcap,3,0.0023,-0.0036,NaN,2,0.4786,39
0,financetoolkit,ft_growth_balance,3,0.0003,-0.0043,NaN,1,0.4933,50
8,fmp,fmp_balance_mcap,3,-0.0236,-0.0302,NaN,0,0.3400,50


{'evaluation_seconds': 3.38, 'feature_horizon_results': 1095}


## Written Analysis


In [6]:
def _fmt_seconds(value: float) -> str:
    return "nan" if pd.isna(value) else f"{value:,.2f}s"

family_counts = feature_metadata.groupby(["source", "family"]).size().rename("feature_count").reset_index().sort_values(["source", "feature_count"], ascending=[True, False])
best_lines = [
    f"- {row.horizon}d: `{row.family}` ({row.source}) mean rank IC {row.mean_rank_ic:.4f}, {int(row.features)} features"
    for row in best_by_horizon.itertuples(index=False)
]
stable_lines = [
    f"- `{row.family}` ({row.source}): avg rank IC {row.avg_rank_ic:.4f}, min rank IC {row.min_rank_ic:.4f}, positive horizons {int(row.positive_horizons)}/{int(row.horizons)}, features {int(row.features)}"
    for row in stable_families.itertuples(index=False)
]
analysis_md = "\n".join([
    f"### {ANALYSIS_LABEL} Fundamental Feature-Family Analysis",
    "",
    f"The capped panel completed on {panel['symbol'].nunique()} symbols, {len(panel):,} symbol-days, {len(feature_cols)} capped features, and {feature_metadata['family'].nunique()} feature families.",
    f"Raw build produced {raw_feature_metadata['feature'].nunique()} candidate features; every family was capped at {config.max_features_per_family} columns using coverage and cross-sectional variance only.",
    f"Raw panel construction took {_fmt_seconds(timings.get('raw_panel_build_seconds', float('nan')))}; rank-IC evaluation took {_fmt_seconds(evaluation_seconds)}; measured core runtime was {_fmt_seconds(timings.get('raw_panel_build_seconds', 0.0) + evaluation_seconds)}.",
    "",
    "#### Family Width After Cap",
    *[f"- `{row.family}` ({row.source}): {int(row.feature_count)} features" for row in family_counts.itertuples(index=False)],
    "",
    "#### Best Family By Horizon",
    *best_lines,
    "",
    "#### Most Stable Families",
    *stable_lines,
    "",
    "#### Interpretation",
    "This notebook isolates fundamental feature families only. FMP daily-adjusted valuation and statement-level market-cap families stay separate from FinanceToolkit-style ratio and growth families. Families that remain stable across horizons are the best first candidates for reusable feature storage; weak families should be split or dropped before training.",
])
display(Markdown(analysis_md))


### OpenBB/FMP screened US >= $1T universe Fundamental Feature-Family Analysis

The capped panel completed on 14 symbols, 27,701 symbol-days, 365 capped features, and 15 feature families.
Raw build produced 376 candidate features; every family was capped at 50 columns using coverage and cross-sectional variance only.
Raw panel construction took 1.23s; rank-IC evaluation took 3.38s; measured core runtime was 4.62s.

#### Family Width After Cap
- `ft_growth_balance` (financetoolkit): 50 features
- `ft_growth_cash` (financetoolkit): 50 features
- `ft_growth_income` (financetoolkit): 38 features
- `ft_ratios_valuation` (financetoolkit): 24 features
- `ft_ratios_solvency` (financetoolkit): 15 features
- `ft_ratios_profitability` (financetoolkit): 14 features
- `ft_ratios_liquidity` (financetoolkit): 7 features
- `ft_ratios_efficiency` (financetoolkit): 5 features
- `fmp_balance_mcap` (fmp): 50 features
- `fmp_cash_mcap` (fmp): 39 features
- `fmp_income_mcap` (fmp): 31 features
- `fmp_daily_mcap_multiple` (fmp): 14 features
- `fmp_daily_mcap_yield` (fmp): 14 features
- `fmp_daily_ev_multiple` (fmp): 7 features
- `fmp_daily_ev_yield` (fmp): 7 features

#### Best Family By Horizon
- 20d: `ft_ratios_valuation` (financetoolkit) mean rank IC 0.0213, 24 features
- 60d: `ft_ratios_valuation` (financetoolkit) mean rank IC 0.0436, 24 features
- 120d: `ft_ratios_valuation` (financetoolkit) mean rank IC 0.0611, 24 features

#### Most Stable Families
- `ft_ratios_valuation` (financetoolkit): avg rank IC 0.0420, min rank IC 0.0213, positive horizons 3/3, features 24
- `ft_ratios_efficiency` (financetoolkit): avg rank IC 0.0264, min rank IC 0.0175, positive horizons 3/3, features 5
- `ft_growth_income` (financetoolkit): avg rank IC 0.0229, min rank IC 0.0130, positive horizons 3/3, features 38
- `ft_ratios_liquidity` (financetoolkit): avg rank IC 0.0166, min rank IC 0.0117, positive horizons 3/3, features 7
- `ft_ratios_profitability` (financetoolkit): avg rank IC 0.0165, min rank IC 0.0061, positive horizons 3/3, features 14
- `fmp_income_mcap` (fmp): avg rank IC 0.0046, min rank IC 0.0001, positive horizons 3/3, features 31
- `ft_growth_cash` (financetoolkit): avg rank IC 0.0072, min rank IC -0.0005, positive horizons 2/3, features 50
- `fmp_cash_mcap` (fmp): avg rank IC 0.0023, min rank IC -0.0036, positive horizons 2/3, features 39
- `ft_growth_balance` (financetoolkit): avg rank IC 0.0003, min rank IC -0.0043, positive horizons 1/3, features 50
- `fmp_balance_mcap` (fmp): avg rank IC -0.0236, min rank IC -0.0302, positive horizons 0/3, features 50
- `fmp_daily_mcap_multiple` (fmp): avg rank IC -0.0324, min rank IC -0.0494, positive horizons 0/3, features 14
- `ft_ratios_solvency` (financetoolkit): avg rank IC -0.0338, min rank IC -0.0538, positive horizons 0/3, features 15
- `fmp_daily_ev_multiple` (fmp): avg rank IC -0.0359, min rank IC -0.0597, positive horizons 0/3, features 7
- `fmp_daily_ev_yield` (fmp): avg rank IC -0.0380, min rank IC -0.0584, positive horizons 0/3, features 7
- `fmp_daily_mcap_yield` (fmp): avg rank IC -0.0449, min rank IC -0.0652, positive horizons 0/3, features 14

#### Interpretation
This notebook isolates fundamental feature families only. FMP daily-adjusted valuation and statement-level market-cap families stay separate from FinanceToolkit-style ratio and growth families. Families that remain stable across horizons are the best first candidates for reusable feature storage; weak families should be split or dropped before training.